# SFT for a native MTP head

Train the floating-point Qwen3.5 native multi-token predictor against assistant
responses while keeping the target model frozen. The default target is a quantized
Qwen3.5-2B; the matching original checkpoint supplies native MTP weights.
This notebook trains **only the MTP head**, then exports it as a standalone drafter.
The other training notebooks keep their original objectives without MTP.

The predictor combines `h[t]` with ground-truth `x[t+1]` to predict `x[t+2]`.
Prompt and padding positions are excluded. This is supervised next-next-token CE;
it does not measure speculative acceptance or guarantee a decoding speedup.


## Setup

Use an Apple Silicon Mac and the Python environment selected by your notebook kernel.
Install once from your checkout, then restart the kernel:

```sh
python -m pip install -e "/absolute/path/to/mlx-vlm[train]" jupyterlab ipykernel
```

Run the cells in order. The first load downloads the model and dataset.


In [ ]:
import json
from dataclasses import asdict
from pathlib import Path

import mlx.core as mx
import mlx.optimizers as optim
import pandas as pd
from datasets import load_dataset
from mlx.utils import tree_flatten

from mlx_vlm import load
from mlx_vlm.prompt_utils import apply_chat_template
from mlx_vlm.speculative.drafters import load_drafter
from mlx_vlm.trainer import Trainer, VLMTrainingArgs, attach_mtp_head, prepare_model_for_training


## 1. Settings

Use the same base/tokenizer family as the head. `MTP_MODEL_ID` can also be a local
floating-point split-head directory. Loading from an original Hub checkpoint may
require downloading its weight shards even though the splitter retains only MTP weights.
Set `BASE_ADAPTER_PATH` if the target already has compatible LoRA/DoRA adapters.
`MTP_CHECKPOINT_PATH` resumes only saved MTP weights; optimizer state starts fresh.


In [ ]:
MODEL_ID = "mlx-community/Qwen3.5-2B-4bit"
MTP_MODEL_ID = "Qwen/Qwen3.5-2B"
DATASET_ID = "mlx-community/JOSIE-v2-Instruct-5K"
BASE_ADAPTER_PATH = None
MTP_CHECKPOINT_PATH = None
OUTPUT_DIR = Path("adapters/vlm/sft_mtp")
EXPORT_DIR = OUTPUT_DIR / "mtp_drafter"
VALIDATION_FRACTION = 0.05
MAX_TRAIN_SAMPLES = 256
MAX_EVAL_SAMPLES = 32
SEED = 42

args = VLMTrainingArgs(
    mtp_loss_weight=1.0,
    iters=100,
    batch_size=1,
    gradient_accumulation_steps=4,
    max_seq_length=1024,
    train_on_completions=True,
    grad_checkpoint=False,
    cache_size=8,
    steps_per_report=10,
    steps_per_eval=50,
    steps_per_save=100,
    adapter_file=str(OUTPUT_DIR / "weights.safetensors"),
)
mx.random.seed(SEED)
args.validate()


## 2. Data

JOSIE already contains `messages`. Remove conversations without a non-empty final
assistant response, then create a seeded training/validation split.


In [ ]:
dataset = load_dataset(DATASET_ID, split="train")
if "messages" not in dataset.column_names:
    raise ValueError(f"Expected a messages column, got {dataset.column_names}.")


def has_assistant_target(row):
    messages = row["messages"]
    return bool(
        messages
        and messages[-1].get("role") == "assistant"
        and isinstance(messages[-1].get("content"), str)
        and messages[-1]["content"].strip()
    )


source_rows = len(dataset)
dataset = dataset.filter(
    has_assistant_target, desc="Keep conversations with assistant targets"
)
dropped_rows = source_rows - len(dataset)
print({"source_rows": source_rows, "dropped_without_assistant_target": dropped_rows})
if len(dataset) < 2:
    raise ValueError("Need at least two conversations with assistant targets to split.")

partitions = dataset.train_test_split(test_size=VALIDATION_FRACTION, seed=SEED)


def limit_rows(rows, maximum):
    if maximum is None:
        return rows
    if maximum < 1:
        raise ValueError("Sample limits must be positive or None.")
    return rows.select(range(min(maximum, len(rows))))


train_rows = limit_rows(partitions["train"], MAX_TRAIN_SAMPLES)
eval_rows = limit_rows(partitions["test"], MAX_EVAL_SAMPLES)

assert len(train_rows) >= args.batch_size, "Reduce batch_size or add training rows."
assert len(eval_rows) >= args.batch_size, "Validation needs a complete batch."
print({"train_rows": len(train_rows), "validation_rows": len(eval_rows)})
print(train_rows.features)

## 3. Freeze the target and attach the native predictor

The base weights, vision tower, shared token embeddings, and output projection
stay frozen. The attached `mtp_head` is the only trainable child. Keep the head
unbound until inference so saved weights do not duplicate the shared base modules.


In [ ]:
model, processor = load(MODEL_ID)
if BASE_ADAPTER_PATH is not None:
    model = prepare_model_for_training(model, checkpoint_path=BASE_ADAPTER_PATH)
model.freeze()
head, kind = load_drafter(MTP_MODEL_ID, kind="mtp")
assert kind == "mtp"
attach_mtp_head(model, head, checkpoint_path=MTP_CHECKPOINT_PATH)
trainable_names = [name for name, _ in tree_flatten(model.trainable_parameters())]
assert trainable_names and all(name.startswith("mtp_head.") for name in trainable_names)
print({"trainable_tensors": len(trainable_names), "trainable_module": "mtp_head"})
formatted_sample = apply_chat_template(
    processor, model.config, train_rows[0]["messages"],
    num_images=0, add_generation_prompt=False,
)
print(formatted_sample[:2000])


## 4. Train and evaluate

The SFT objective reports base next-token NLL plus weighted MTP CE. Base NLL is
constant with respect to the head-only parameters; follow `mtp_loss` to assess
head fitting. Held-out MTP loss is a training diagnostic, not a speed benchmark.


In [ ]:
optimizer = optim.AdamW(learning_rate=1e-4, weight_decay=0.0)
trainer = Trainer(
    model, args, optimizer=optimizer, algorithm="sft", processing_class=processor,
    train_dataset=train_rows, eval_dataset=eval_rows, seed=SEED,
)
baseline = trainer.evaluate()
metrics = trainer.train()
validation = trainer.evaluate()
assert mx.isfinite(mx.array(validation["val_mtp_loss"])).item()
pd.DataFrame([baseline, validation], index=["before", "after"]).reindex(
    columns=["val_mtp_loss", "val_num_mtp_tokens", "val_nll", "val_loss"]
)


## 5. Save and verify a training checkpoint

`weights.safetensors` contains only `mtp_head.*` tensors in this head-only setup.
Reload the matching native head and apply those tensors with `attach_mtp_head`.
The frozen target is reused here; reload it and any base adapters in a fresh session.


In [ ]:
trainer.save_adapter()
saved_weights = mx.load(args.adapter_file)
assert saved_weights and all(name.startswith("mtp_head.") for name in saved_weights)
reloaded_head, _ = load_drafter(MTP_MODEL_ID, kind="mtp")
attach_mtp_head(model, reloaded_head, checkpoint_path=args.adapter_file)
reloaded_validation = trainer.evaluate()
assert mx.allclose(
    mx.array(validation["val_mtp_loss"]), mx.array(reloaded_validation["val_mtp_loss"]),
    rtol=1e-5, atol=1e-5,
).item()


## 6. Export a standalone drafter

Export sanitized, unprefixed native head weights and its configuration. This is
a head artifact: pair it with the exact frozen target and any original base adapters.
The tokenizer stays with the target. The final cell reloads the exported artifact.


In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
export_config = asdict(model.mtp_head.config)
export_config["text_config"]["mtp_num_hidden_layers"] = len(model.mtp_head.layers)
(EXPORT_DIR / "config.json").write_text(json.dumps(export_config, indent=2) + "\n", encoding="utf-8")
mx.save_safetensors(
    str(EXPORT_DIR / "model.safetensors"),
    dict(tree_flatten(model.mtp_head.parameters())),
)
(EXPORT_DIR / "training_info.json").write_text(
    json.dumps({
        "target_model": MODEL_ID, "base_adapter_path": str(BASE_ADAPTER_PATH) if BASE_ADAPTER_PATH else None,
        "source_mtp_model": MTP_MODEL_ID, "dataset": DATASET_ID,
        "seed": SEED, "training_args": asdict(args), "head_only": True,
    }, indent=2) + "\n", encoding="utf-8",
)
exported_head, exported_kind = load_drafter(str(EXPORT_DIR), kind="mtp")
assert exported_kind == "mtp"
for name, value in tree_flatten(exported_head.parameters()):
    assert mx.allclose(value, dict(tree_flatten(model.mtp_head.parameters()))[name]).item()
print(EXPORT_DIR.resolve())


## 7. Use the trained head for speculative decoding

For the default frozen target (no base adapters), run:

```sh
python -m mlx_vlm.generate \
  --model mlx-community/Qwen3.5-2B-4bit \
  --draft-model ./adapters/vlm/sft_mtp/mtp_drafter \
  --draft-block-size 3 --prompt "Explain gradient accumulation." \
  --max-tokens 128 --temperature 0
```

If `BASE_ADAPTER_PATH` was set, restore those same base adapters for inference too.
Compare generated quality, acceptance rate, memory, and throughput against your
original head on representative held-out prompts. See the [training guide](../../docs/training.md).
